# RAG Retrieval Accuracy — Raw Post Index

Measures how accurately the **raw-text FAISS index** retrieves training essays
whose label matches the query essay's true label, evaluated on the **val split**.

**Pipeline:**
1. Load `data/split/essays/val.csv` (247 rows).
2. Embed each val essay with the raw-text embedder (`_embed_single`).
3. For each trait and k ∈ {3, 5}: retrieve top-k from `data/vector_db/essays_rawpost/`.
4. Count how many retrieved samples share the query's true label (**label match rate**).
5. Save results to `result/retrieve_accuracy/rawpost/`.

**Metrics per (trait, k):**
- `mean_match_rate` — average fraction of retrieved samples whose label matches the query.
- `std_match_rate`  — standard deviation across queries.
- `hit_rate`        — fraction of queries where ≥ 1 retrieved sample matches.

In [1]:
import os, sys, time
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd().resolve()
if not (project_root / "ptd_model").exists():
    project_root = (project_root / ".." / "..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: F:\std\GR\code\model_x_ocean


## Configuration

In [2]:
VAL_CSV    = str(project_root / "data/split/essays/val.csv")
RAWPOST_DB = str(project_root / "data/vector_db/essays_rawpost")
RES_DIR    = str(project_root / "result/retrieve_accuracy/rawpost")

K_VALUES = [3, 5]

TRAITS = {
    "cOPN": "Openness to Experience",
    "cCON": "Conscientiousness",
    "cEXT": "Extraversion",
    "cAGR": "Agreeableness",
    "cNEU": "Neuroticism",
}

val_df = pd.read_csv(VAL_CSV)
print(f"Val split : {len(val_df)} rows")
print(f"Rawpost DB: {RAWPOST_DB}")
print(f"Output    : {RES_DIR}")

for p in (RAWPOST_DB,):
    if not Path(p).exists():
        raise RuntimeError(f"Missing: {p}. Run notebook/data_process/build_rawpost_index.ipynb first.")

Val split : 247 rows
Rawpost DB: F:\std\GR\code\model_x_ocean\data\vector_db\essays_rawpost
Output    : F:\std\GR\code\model_x_ocean\result\retrieve_accuracy\rawpost


## Step 1 — Load index & embed val essays

In [3]:
from rag.embedder import _embed_single
from rag.faiss_index import FAISSIndex

rawpost_index = FAISSIndex(dimension=0)
rawpost_index.load(
    str(Path(RAWPOST_DB) / "vectors.faiss"),
    str(Path(RAWPOST_DB) / "vectors_meta.jsonl"),
)
print(f"Rawpost index loaded: {rawpost_index._index.ntotal} vectors, dim={rawpost_index.dimension}")

Rawpost index loaded: 1974 vectors, dim=768


In [4]:
print(f"Embedding {len(val_df)} val essays (raw text) ...")
t0 = time.time()
query_embs = []
for i, (_, row) in enumerate(val_df.iterrows()):
    vec = _embed_single(str(row["text"]))
    query_embs.append(vec)
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(val_df)}  ({time.time()-t0:.0f}s)")

query_embs = np.array(query_embs, dtype="float32")
print(f"Done in {time.time()-t0:.1f}s.  Shape: {query_embs.shape}")

Embedding 247 val essays (raw text) ...
[embedder] Loading embedding model: nomic-ai/nomic-embed-text-v1.5


<All keys matched successfully>


  50/247  (124s)
  100/247  (247s)
  150/247  (341s)
  200/247  (445s)
Done in 532.8s.  Shape: (247, 768)


## Step 2 — Evaluate retrieval accuracy (one trait at a time, k = 3 and k = 5)

In [5]:
def normalize_label(val):
    s = str(val).strip().lower()
    if s in ("1", "1.0"): return "high"
    if s in ("0", "0.0"): return "low"
    return s


all_rows     = []
per_query_rows = []

for k in K_VALUES:
    print(f"\n{'='*55}")
    print(f"  k = {k}")
    print(f"{'='*55}")

    for trait_code, trait_full in TRAITS.items():
        match_rates, hits = [], []

        for i, (_, row) in enumerate(val_df.iterrows()):
            true_label = normalize_label(row[trait_code])
            if true_label not in ("high", "low"):
                continue

            # Retrieve k * 6 candidates, filter to those with a label for this trait
            _, results = rawpost_index.search(query_embs[i], k * 6)
            filtered   = [
                r for r in results
                if trait_full in r.get("trait_labels", {})
            ][:k]

            n       = len(filtered)
            matches = sum(
                1 for r in filtered
                if r["trait_labels"][trait_full] == true_label
            )
            rate = matches / n if n > 0 else 0.0
            hit  = int(matches > 0)

            match_rates.append(rate)
            hits.append(hit)
            per_query_rows.append({
                "query_idx":   i,
                "trait":       trait_full,
                "k":           k,
                "true_label":  true_label,
                "n_retrieved": n,
                "match_count": matches,
                "match_rate":  rate,
                "hit":         hit,
            })

        mean_r = float(np.mean(match_rates))
        std_r  = float(np.std(match_rates))
        hit_r  = float(np.mean(hits))
        print(f"  {trait_full:<30s}  match_rate={mean_r:.4f} ± {std_r:.4f}  hit_rate={hit_r:.4f}")

        all_rows.append({
            "trait":           trait_full,
            "k":               k,
            "n_queries":       len(match_rates),
            "mean_match_rate": mean_r,
            "std_match_rate":  std_r,
            "hit_rate":        hit_r,
        })

summary_df   = pd.DataFrame(all_rows)
per_query_df = pd.DataFrame(per_query_rows)
print("\nEvaluation complete.")


  k = 3
  Openness to Experience          match_rate=0.5020 ± 0.2906  hit_rate=0.8826
  Conscientiousness               match_rate=0.5236 ± 0.2802  hit_rate=0.9069
  Extraversion                    match_rate=0.5452 ± 0.3140  hit_rate=0.8664
  Agreeableness                   match_rate=0.5169 ± 0.2870  hit_rate=0.8907
  Neuroticism                     match_rate=0.5439 ± 0.2904  hit_rate=0.8988

  k = 5
  Openness to Experience          match_rate=0.5150 ± 0.2299  hit_rate=0.9717
  Conscientiousness               match_rate=0.5109 ± 0.2126  hit_rate=0.9798
  Extraversion                    match_rate=0.5312 ± 0.2339  hit_rate=0.9798
  Agreeableness                   match_rate=0.5198 ± 0.2009  hit_rate=0.9879
  Neuroticism                     match_rate=0.5320 ± 0.2358  hit_rate=0.9555

Evaluation complete.


## Step 3 — Display & save results

In [6]:
print("=== Mean label match rate (fraction of retrieved samples matching query's true label) ===")
pivot = summary_df.pivot_table(
    index="trait", columns="k", values="mean_match_rate"
).rename_axis(columns="k")
display(pivot.round(4))

print("\n=== Hit rate (fraction of queries where ≥1 retrieved sample matches) ===")
pivot_hit = summary_df.pivot_table(
    index="trait", columns="k", values="hit_rate"
).rename_axis(columns="k")
display(pivot_hit.round(4))

=== Mean label match rate (fraction of retrieved samples matching query's true label) ===


k,3,5
trait,,
Agreeableness,0.5169,0.5198
Conscientiousness,0.5236,0.5109
Extraversion,0.5452,0.5312
Neuroticism,0.5439,0.5320
Openness to Experience,0.5020,0.5150



=== Hit rate (fraction of queries where ≥1 retrieved sample matches) ===


k,3,5
trait,,
Agreeableness,0.8907,0.9879
Conscientiousness,0.9069,0.9798
Extraversion,0.8664,0.9798
Neuroticism,0.8988,0.9555
Openness to Experience,0.8826,0.9717


In [ ]:
os.makedirs(RES_DIR, exist_ok=True)

summary_path   = os.path.join(RES_DIR, "accuracy_summary.csv")
per_query_path = os.path.join(RES_DIR, "per_query_details.csv")

summary_df.to_csv(summary_path, index=False)
per_query_df.to_csv(per_query_path, index=False)

print(f"Saved summary      -> {summary_path}")
print(f"Saved per-query    -> {per_query_path}")
print(f"\nRows in summary    : {len(summary_df)}")
print(f"Rows in per-query  : {len(per_query_df)}")

Saved summary      -> F:\std\GR\code\model_x_ocean\result\retrieve_accuracy\rawpost\accuracy_summary.csv
Saved per-query    -> F:\std\GR\code\model_x_ocean\result\retrieve_accuracy\rawpost\per_query_details.csv

Rows in summary    : 10
Rows in per-query  : 2470


: 